# Stage 1: Experimental — VWAP-based wallet preselection

Preselect larger wallet sets that are profitable (average_roi > 0.02), then
compute per-trade trailing 15-minute VWAP and VWAP volume for BUY trades
by (wallet, condition_id, token_id).

These VWAP features are added back to the trade DataFrames for downstream
analysis.

**Output:** `stage1_experimental_result.json` with preselected wallets + VWAP stats.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from IPython.display import display

from lib import (
    load_trades,
    compute_copyable_notional,
    compute_opening_metrics,
    DEFAULT_TAGS,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics


## Parameters

In [2]:
MIN_ROI = 0.03
VWAP_WINDOW_MINUTES = 15

print(f"MIN_ROI: {MIN_ROI}")
print(f"VWAP_WINDOW_MINUTES: {VWAP_WINDOW_MINUTES}")

MIN_ROI: 0.03
VWAP_WINDOW_MINUTES: 15


## Load data

Markets: 1974837


Filtered markets for {'Weather'}: 91123
Loading 16 trade shards...


Total trades loaded: 14,250,603
Unique wallets: 4,082
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-27 06:12:25+00:00


Split by trade date:
  Train:  6,035,492 trades  (23,237 markets)  < 2026-06-01
  Val:    4,766,255 trades  (19,787 markets)  2026-06-01 .. 2026-07-01


  Test:   3,448,856 trades  (16,333 markets)  >= 2026-07-01


  Total: 14,250,603 trades  (56,875 markets)



  Markets overlapping train/val: 1350
  Markets overlapping train/test: 1
  Markets overlapping val/test: 1132


## Compute wallet metrics on training data

In [4]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "sell_roi", "copyable_pnl", "copyable_roi", "num_buckets"]].head(10)

Wallets with metrics: 3584


,wallet,buy_roi,sell_roi,copyable_pnl,copyable_roi,num_buckets
0,0x00546dfeb4097e232c86a775ccbe3c9b84c0cab1,0.023563,NaN,12.845816,0.037440,53
1,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.005054,0.002003,-3.331733,0.000000,235
2,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.026175,0.036027,227.315103,0.011217,5511
3,0x00833cc2d777e6f2fc8437679124024ae6468cb1,NaN,NaN,0.000000,NaN,2
4,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.677602,-1.000000,119.386767,0.504015,50
5,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,NaN,NaN,-20.495392,NaN,344
6,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.012020,-1.000000,12.681270,-0.101257,81
7,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.025917,-0.181527,136.243576,0.005965,5369
8,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.196781,-0.063492,-316.535035,0.000000,1985
9,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,0.005820,-0.969739,24.241672,0.005206,2999


## Preselect wallets by average buy ROI

In [5]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='str')

In [6]:
preselected_ws = set(
    wallet_vol.loc[wallet_vol["buy_roi"] > MIN_ROI, "wallet"]
)
print(f"Preselected wallets (buy_roi > {MIN_ROI}): {len(preselected_ws)}")

# Quick stats on the preselected set
preselected_df = wallet_vol[wallet_vol["wallet"].isin(preselected_ws)].copy()
print()
print(f"  Buy ROI range:  {preselected_df['buy_roi'].min():.4f} — {preselected_df['buy_roi'].max():.4f}")
print(f"  Avg num_buckets: {preselected_df['num_buckets'].mean():.0f}")
print(f"  total_pnl: ${preselected_df['total_pnl'].sum():,.0f}")
print(f"  trades: {preselected_df['trade_count'].sum():,.0f}")

Preselected wallets (buy_roi > 0.03): 1441

  Buy ROI range:  0.0304 — 199.0000
  Avg num_buckets: 1329
  total_pnl: $1,123,268
  trades: 2,164,403


In [7]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='str')

## Compute 15-min trailing VWAP for BUY trades

For each BUY trade by a preselected wallet, look back 15 minutes over
the same (wallet, condition_id, token_id) and compute:
- **vwap_15m**: volume-weighted average price of trades **strictly before** this one
- **vwap_volume_15m**: total USDC volume of trades in the window

> `TEST_MODE=True` limits to 50 wallets for quick validation.

In [8]:
# buy_mask = df_full["wallet"].isin(preselected_ws) & (df_full["side"] == "BUY") 
# buy_trades = df_full.loc[buy_mask].copy() 
# print(f"BUY trades by preselected wallets: {len(buy_trades):,}")

# buy_trades = buy_trades.sort_values(
#     ["wallet", "condition_id", "token_id", "dt"],
#     kind="mergesort"
# ).reset_index(drop=True)

# buy_trades["vwap_15m"] = np.nan
# buy_trades["vwap_volume_15m"] = 0.0

# window_ns = np.timedelta64(VWAP_WINDOW_MINUTES, "m")

# for _, idx in buy_trades.groupby(
#     ["wallet", "condition_id", "token_id"],
#     sort=False
# ).groups.items():

#     g = buy_trades.loc[idx]

#     t = g["dt"].values
#     qty = g["quantity"].to_numpy()
#     usdc = g["usdc_amount"].to_numpy()
#     pq = (g["price"] * g["quantity"]).to_numpy()

#     left = 0
#     sum_qty = 0.0
#     sum_pq = 0.0
#     sum_usdc = 0.0

#     out_vwap = np.empty(len(g))
#     out_vol = np.empty(len(g))

#     for i in range(len(g)):

#         # remove expired trades
#         while left < i and t[left] < t[i] - window_ns:
#             sum_qty -= qty[left]
#             sum_pq -= pq[left]
#             sum_usdc -= usdc[left]
#             left += 1

#         # current trade is excluded
#         out_vwap[i] = np.nan if sum_qty == 0 else sum_pq / sum_qty
#         out_vol[i] = sum_usdc

#         # add current trade
#         sum_qty += qty[i]
#         sum_pq += pq[i]
#         sum_usdc += usdc[i]

#     buy_trades.loc[idx, "vwap_15m"] = out_vwap
#     buy_trades.loc[idx, "vwap_volume_15m"] = out_vol

# df_full = df_full.merge(
#     buy_trades[["tx_hash","vwap_15m","vwap_volume_15m"]],
#     on="tx_hash",
#     how="left",
# )

# df_train = df_full[df_full["dt"] < train_cutoff].copy()
# df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
# df_test = df_full[df_full["dt"] >= val_cutoff].copy()

In [9]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='str')

In [10]:
bad_buy_leaders = wallet_vol[
    (wallet_vol['buy_roi'] >= 0.04)
    & (wallet_vol['trade_count'] >= 1000)
    & (wallet_vol['total_pnl'] > 10000)
    & (wallet_vol['max_drawdown_to_pnl'] <= 0.5)
    & (wallet_vol['copyable_roi'] < 0.02)
    & (wallet_vol['buy_copyable_pnl'] * -1 >= 1000)
]

print(f"Bad buy leaders: {len(bad_buy_leaders)}")
print(f"Bad buy leader train trades: {len(df_train[df_train['wallet'].isin(bad_buy_leaders['wallet'])])}")
print(f"Bad buy leader pnl: ${bad_buy_leaders['total_pnl'].sum():,.0f}")
print(f"Bad buy leader copyable buy pnl: ${bad_buy_leaders['buy_copyable_pnl'].sum():,.0f}")
print(f"Bad buy leader buy roi: ${bad_buy_leaders['buy_pnl'].sum() / bad_buy_leaders['buy_notional'].sum():,.2f}")
print(f"Bad buy leader val pnl: ${df_val[df_val['wallet'].isin(bad_buy_leaders['wallet'])]['pnl'].sum():,.0f}")

Bad buy leaders: 2
Bad buy leader train trades: 133447
Bad buy leader pnl: $29,390
Bad buy leader copyable buy pnl: $-11,750
Bad buy leader buy roi: $0.07
Bad buy leader val pnl: $9,746


In [11]:
bad_buy_leaders.head()

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,return,copyable_pnl_factor,copyable_roi,opening_pnl,opening_notional,opening_copyable_pnl,opening_buys,opening_copyable_notional,opening_roi,opening_copyable_roi
1153,0xb06a0eae498750ed0acac7e1f759f741c56e52f5,0.322946,92384,10339,103599,263354.778806,15099.549014,-9365.720930,0.275840,0.435316,...,0.057335,0.0,0.0,1351.187818,17103.279037,-271.397678,10298.0,6479.346074,0.079002,-0.041887
1289,0xc7d02944a76b9f83b199e9090ecc92c82d241f8a,0.434007,28639,4309,29848,165292.295427,14290.413849,-3584.065829,0.300944,0.409550,...,0.086455,0.0,0.0,3230.185197,43866.304970,-1175.765783,5857.0,10039.604894,0.073637,-0.117113


In [12]:
wallet_vol.columns

Index(['wallet', 'pnl_volatility', 'num_buckets', 'num_markets', 'trade_count',
       'total_notional', 'total_pnl', 'copyable_pnl', 'top5_pnl_pct',
       'top10_pnl_pct', 'worst5_pnl_pct', 'top_market_pnl_pct',
       'top_market_abs_pnl_pct', 'market_pnl_hhi', 'positive_bucket_share',
       'median_roi', 'median_dt', 'average_roi', 'buy_roi', 'buy_pnl',
       'buy_notional', 'buy_copyable_pnl', 'sell_pnl', 'sell_roi',
       'max_drawdown', 'max_drawdown_to_pnl', 'max_copyable_drawdown',
       'max_copyable_drawdown_to_copyable_pnl', 'return',
       'copyable_pnl_factor', 'copyable_roi', 'opening_pnl',
       'opening_notional', 'opening_copyable_pnl', 'opening_buys',
       'opening_copyable_notional', 'opening_roi', 'opening_copyable_roi'],
      dtype='str')

In [13]:
(
    wallet_vol[
        (wallet_vol['trade_count'] >= 500)
        # & (wallet_vol['average_roi'] >= 0.05)
        # & (wallet_vol['copyable_pnl'] >= 100)
        ]
    ).sort_values('total_pnl', ascending=False).head(20)

,wallet,pnl_volatility,num_buckets,num_markets,trade_count,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,...,return,copyable_pnl_factor,copyable_roi,opening_pnl,opening_notional,opening_copyable_pnl,opening_buys,opening_copyable_notional,opening_roi,opening_copyable_roi
1149,0xafde461fce5aa0fabdb7711c59db93b65e343e1d,0.957756,3163,394,3718,1.092031e+05,30043.942187,11841.913827,1.032128,1.162751,...,0.275120,0.394153,0.487646,3969.194801,6349.977287,513.202781,446.0,1857.731022,0.625072,0.276252
1143,0xae2e04fe9d8ccba5e45ba17ddf9dfbef498c40ad,1.857969,3859,484,4716,1.033217e+05,22667.525475,2248.495040,1.158076,1.258107,...,0.219388,0.099195,0.065497,4883.986659,10320.547633,-913.980441,580.0,3223.326167,0.473229,-0.283552
1173,0xb40e89677d59665d5188541ad860450a6e2a7cc9,0.097495,266467,7853,286565,1.211482e+06,18046.306392,-23867.685176,0.106982,0.180791,...,0.014896,0.000000,0.000000,2189.775995,210062.939829,-3181.590138,46879.0,82367.297536,0.010424,-0.038627
498,0x488c725253fc21c7a9ca812030dc2f6343f98c1c,1.609136,2359,391,3269,1.861033e+05,16867.802367,2569.051445,0.732761,0.990217,...,0.090637,0.152305,0.089965,8224.460232,25504.823979,371.075386,475.0,3683.864651,0.322467,0.100730
1153,0xb06a0eae498750ed0acac7e1f759f741c56e52f5,0.322946,92384,10339,103599,2.633548e+05,15099.549014,-9365.720930,0.275840,0.435316,...,0.057335,0.000000,0.000000,1351.187818,17103.279037,-271.397678,10298.0,6479.346074,0.079002,-0.041887
1289,0xc7d02944a76b9f83b199e9090ecc92c82d241f8a,0.434007,28639,4309,29848,1.652923e+05,14290.413849,-3584.065829,0.300944,0.409550,...,0.086455,0.000000,0.000000,3230.185197,43866.304970,-1175.765783,5857.0,10039.604894,0.073637,-0.117113
947,0x8fb431f5057112cfdb0e7f566b4b687cb69cb9ed,0.786399,3379,358,4248,8.515574e+04,12334.921901,3913.186756,0.431021,0.664747,...,0.144851,0.317245,0.035503,2281.637325,12308.219513,1063.131160,413.0,3591.994193,0.185375,0.295972
927,0x8d0930676d559cc8fb7d8af0c555791c1820143f,0.112830,5356,697,5896,1.150638e+06,11769.118661,2002.084483,0.261282,0.403760,...,0.010228,0.170113,0.089243,4130.966471,245683.787199,849.066007,711.0,10868.312838,0.016814,0.078123
1585,0xf1e18ec32b2f1e123bc098e3956e6fd00012c152,1.729986,1437,155,2745,3.164178e+04,11222.055485,2332.129447,0.696916,0.872260,...,0.354659,0.207817,0.436024,1971.908374,2595.435486,436.082675,166.0,640.238506,0.759760,0.681125
1410,0xdafdc201e4a769c4424462f9210763b221da2ad4,2.106591,1691,278,2174,9.428722e+04,10559.840084,943.980646,1.078021,1.280717,...,0.111997,0.089393,0.004235,2816.186397,27036.603046,856.983938,290.0,3244.420900,0.104162,0.264141


In [14]:
if 'bad_leader_wallet' not in df_full.columns:
    bad_buy_leader_trades = df_full[(df_full['wallet'].isin(bad_buy_leaders['wallet'])) & (df_full['side'] == 'BUY')]
    print(f"Bad buy leader full trades: {len(bad_buy_leader_trades)}")

    leaders = bad_buy_leader_trades.rename(columns={"dt": "dt_leader", "wallet": "bad_leader_wallet"})[['dt_leader', 'bad_leader_wallet', 'condition_id', 'outcome']]

    df_full = pd.merge_asof(
        df_full.sort_values("dt"),
        leaders.sort_values("dt_leader"),
        left_on="dt",
        right_on="dt_leader",
        by=["condition_id", "outcome"],
        direction="backward",
        tolerance=pd.Timedelta(minutes=5),
        allow_exact_matches=False,
)

Bad buy leader full trades: 171233


In [15]:
df_train = df_full[df_full["dt"] < train_cutoff].copy()
df_val = df_full[(df_full["dt"] >= train_cutoff) & (df_full["dt"] < val_cutoff)].copy()
df_test = df_full[df_full["dt"] >= val_cutoff].copy()

print(f"Bad leader trades in train: {len(df_train[df_train['bad_leader_wallet'].notnull()])}")
print(f"Bad leader trades in val: {len(df_val[df_val['bad_leader_wallet'].notnull()])}")
print(f"Bad leader trades in test: {len(df_test[df_test['bad_leader_wallet'].notnull()])}")

Bad leader trades in train: 310208
Bad leader trades in val: 114080


Bad leader trades in test: 53493


## Signal Quality Framework

We evaluate each signal using **Information Coefficient (IC)** and
**Information Ratio (IR)**, following Grinold & Kahn (1999),
*Active Portfolio Management* (McGraw-Hill).

| Metric | Definition | Interpretation |
|--------|------------|----------------|
| **IC** | Spearman rank correlation between signal value and forward copyable ROI | Does a higher signal predict better PnL? |
| **IR** | Mean(IC) / Std(IC) across daily chunks | How consistent is the predictive power? |
| **Hit Rate** | % of events where signal sign matches PnL sign | Directional accuracy |
| **Bootstrap CI** | 2.5th-97.5th percentile of IC over 10k resamples | Is IC sign reliably non-zero? |

Signal overlap is measured with **coincidence rate** (do they fire together?)
and **IC correlation** (are their predictions redundant?).

In [16]:

# Signal quality: IC, IR, bootstrap, overlap, combination

import numpy as np
import pandas as pd


def _rankdata(v):
    """Fractional ranking (scipy.stats.rankdata, method='average')."""
    n = len(v)
    sorter = np.argsort(v, kind="mergesort")
    ordinal = np.empty(n, dtype=np.intp)
    ordinal[sorter] = np.arange(n)
    inv = np.argsort(sorter, kind="mergesort")
    rank = ordinal + 1.0
    i = 0
    while i < n:
        j = i + 1
        while j < n and v[sorter[j]] == v[sorter[i]]:
            j += 1
        if j > i + 1:
            avg_rank = (i + j + 1) / 2.0
            for k in range(i, j):
                rank[sorter[k]] = avg_rank
        i = j
    return rank[inv]


def spearman_rho(x, y):
    """Spearman rank correlation (numpy-only)."""
    mask = np.isfinite(x) & np.isfinite(y)
    n = mask.sum()
    if n < 10:
        return np.nan
    rx = _rankdata(x[mask].values if hasattr(x, 'values') else x[mask])
    ry = _rankdata(y[mask].values if hasattr(y, 'values') else y[mask])
    rx_m = rx.mean()
    ry_m = ry.mean()
    num = np.sum((rx - rx_m) * (ry - ry_m))
    den = np.sqrt(np.sum((rx - rx_m)**2) * np.sum((ry - ry_m)**2))
    return num / den if den != 0 else np.nan


def compute_event_ic(signal, forward_roi):
    """IC: rank correlation between signal and forward copyable ROI."""
    return spearman_rho(signal, forward_roi)


def compute_event_ir(signal, forward_roi, timestamps, freq="D"):
    """IR = mean(IC_chunk) / std(IC_chunk) across time chunks.
    
    Higher IR means predictive power is consistent (Grinold & Kahn Ch. 7).
    """
    ts = timestamps
    chunks = pd.Series(index=pd.DatetimeIndex(ts), data=np.arange(len(signal))).groupby(
        pd.Grouper(freq=freq)
    )
    ics = []
    for _, idx in chunks:
        if len(idx) < 5:
            continue
        rho = compute_event_ic(signal.iloc[idx], forward_roi.iloc[idx])
        if not np.isnan(rho):
            ics.append(rho)
    if len(ics) < 3:
        return np.nan
    arr = np.array(ics)
    return float(arr.mean() / arr.std(ddof=1)) if arr.std(ddof=1) > 0 else np.nan


def bootstrap_ic(signal, forward_roi, n_iter=10_000, alpha=0.05, seed=42):
    """Bootstrap CI for IC (Efron & Tibshirani 1993).
    
    Returns (mean_ic, ci_lower, ci_upper).
    """
    mask = signal.notna() & forward_roi.notna()
    s = signal[mask].values
    p = forward_roi[mask].values
    n = len(s)
    if n < 10:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot_ics = np.empty(n_iter)
    for i in range(n_iter):
        idx = rng.integers(0, n, n)
        boot_ics[i] = spearman_rho(s[idx], p[idx])
    mean_ic = float(np.nanmean(boot_ics))
    ci_lo = float(np.nanpercentile(boot_ics, 100 * alpha / 2))
    ci_hi = float(np.nanpercentile(boot_ics, 100 * (1 - alpha / 2)))
    return mean_ic, ci_lo, ci_hi


def hit_rate(signal, forward_roi):
    """Fraction of events where signal sign matches PnL sign."""
    mask = signal.notna() & forward_roi.notna() & (forward_roi != 0)
    if mask.sum() < 10:
        return np.nan
    sgn_sig = np.sign(signal[mask])
    sgn_pnl = np.sign(forward_roi[mask])
    return float((sgn_sig == sgn_pnl).mean())


def signal_quality_report(signals_df, signal_cols, roi_col="copyable_roi",
                           dt_col="dt", ir_freq="D", n_bootstrap=5_000):
    """Compute IC, IR, hit rate, and bootstrap CI for each signal.
    
    Returns DataFrame with one row per signal.
    """
    rows = []
    for col in signal_cols:
        ic = compute_event_ic(signals_df[col], signals_df[roi_col])
        ir = compute_event_ir(signals_df[col], signals_df[roi_col],
                               signals_df[dt_col], freq=ir_freq)
        hr = hit_rate(signals_df[col], signals_df[roi_col])
        m, clo, chi = bootstrap_ic(signals_df[col], signals_df[roi_col],
                                    n_iter=n_bootstrap)
        rows.append({
            "signal": col,
            "IC": ic,
            "IR": ir,
            "hit_rate": hr,
            "bootstrap_mean_ic": m,
            "bootstrap_ci_lo": clo,
            "bootstrap_ci_hi": chi,
            "n_events": int(signals_df[col].notna().sum()),
        })
    return pd.DataFrame(rows).sort_values("IC", ascending=False, key=abs)


# === Signal overlap ===

def coincidence_rate(s1, s2):
    """Jaccard-like coincidence: P(both non-zero | either non-zero)."""
    both = ((s1.notna() & (s1 != 0)) & (s2.notna() & (s2 != 0))).sum()
    either = ((s1.notna() & (s1 != 0)) | (s2.notna() & (s2 != 0))).sum()
    return both / either if either > 0 else 0.0


def ic_correlation_matrix(signals_df, signal_cols, roi_col="copyable_roi"):
    """Pairwise IC of signal values on overlapping events."""
    n = len(signal_cols)
    mat = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            if i == j:
                mat[i, j] = 1.0
                continue
            both = signals_df[signal_cols[i]].notna() & signals_df[signal_cols[j]].notna()
            if both.sum() < 10:
                continue
            mat[i, j] = compute_event_ic(
                signals_df.loc[both, signal_cols[i]],
                signals_df.loc[both, signal_cols[j]],
            )
    return pd.DataFrame(mat, index=signal_cols, columns=signal_cols)


# === Signal combination ===

def compute_optimal_weights(
    signals_df, signal_cols, roi_col="copyable_roi",
    shrinkage=0.5,
):
    """Markowitz-optimal signal weights with shrinkage (Grinold & Kahn Ch. 13).
    
    w = (1-lambda) * inv(Sigma) * IC + lambda * (1/n)
    
    Parameters
    ----------
    shrinkage : float
        0 = full Markowitz, 1 = equal weight.
    
    Returns
    -------
    pd.Series of weights indexed by signal_cols.
    """
    n = len(signal_cols)
    ic_vec = np.array([
        compute_event_ic(signals_df[c], signals_df[roi_col]) or 0.0
        for c in signal_cols
    ])
    
    valid = signals_df[signal_cols].notna().all(axis=1)
    if valid.sum() < 10 or n <= 1:
        return pd.Series(np.ones(n) / n, index=signal_cols)
    
    sig_vals = signals_df.loc[valid, signal_cols].values
    cov = np.cov(sig_vals, rowvar=False)
    avg_var = np.trace(cov) / n
    shrunk_cov = (1 - shrinkage) * cov + shrinkage * np.eye(n) * avg_var
    
    try:
        inv_cov = np.linalg.solve(shrunk_cov, np.eye(n))
        w = inv_cov @ ic_vec
        w_abs_sum = np.sum(np.abs(w))
        if w_abs_sum > 1e-12:
            w = w / w_abs_sum
        else:
            w = np.ones(n) / n
    except np.linalg.LinAlgError:
        w = np.ones(n) / n
    
    return pd.Series(w, index=signal_cols)


def apply_composite_score(signals_df, signal_cols, weights):
    """Composite signal = sum w_i * signal_i."""
    result = np.zeros(len(signals_df))
    for col in signal_cols:
        result += weights[col] * signals_df[col].fillna(0.0).values
    return pd.Series(result, index=signals_df.index)


def cs_rank(s, grouper=None):
    """Cross-sectional rank transform. Maps values to [-1, 1] within groups.
    
    If grouper is provided, ranks within each group independently.
    Standard Grinold & Kahn normalization.
    """
    if grouper is not None:
        result = s.groupby(grouper, sort=False).transform(
            lambda x: 2.0 * (_rankdata(x.values) - 1.0) / max(len(x) - 1, 1) - 1.0
        )
    else:
        n = len(s)
        r = _rankdata(s.values) if hasattr(s, 'values') else _rankdata(np.asarray(s))
        result = 2.0 * (r - 1.0) / max(n - 1, 1) - 1.0
    return result


## Stage 2: Signal Discovery

**New approach:** Test signals on a broad random sample of trades first (all wallets).
Only apply surviving signals to copy-universe trades.

Key changes vs Stage 1:
- Continuous wallet quality scores (instead of binary filters)
- Market-level signals from ALL trades (not just quality wallets)
- Forward selection: add signals conditionally
- Broad sample = 200K random BUY trades (much higher statistical power)

## Parameters

In [17]:

import numpy as np
import pandas as pd

# === Discovery phase params ===
MIN_WALLET_TRADES = 100
BROAD_SAMPLE_SIZE = 200_000
SIGNAL_WINDOW_MIN = 15
VWAP_BUCKET_MIN = 5

# === Quality wallet scoring ===
QUALITY_METRICS = ['copyable_roi', 'buy_roi', 'positive_bucket_share']
QUALITY_PENALTIES = ['max_drawdown_to_pnl', 'pnl_volatility', 'top_market_pnl_pct']

# === Quality wallet threshold (pct of wallets) ===
QW_TOP_PCT = 0.20  # top 20% = quality wallets

print(f"MIN_WALLET_TRADES={MIN_WALLET_TRADES}")
print(f"BROAD_SAMPLE_SIZE={BROAD_SAMPLE_SIZE:,}")
print(f"SIGNAL_WINDOW_MIN={SIGNAL_WINDOW_MIN}")
print(f"QW_TOP_PCT={QW_TOP_PCT}")


MIN_WALLET_TRADES=100
BROAD_SAMPLE_SIZE=200,000
SIGNAL_WINDOW_MIN=15
QW_TOP_PCT=0.2


In [18]:

# Load trades + wallet metrics (from stage1's first 21 cells)
# wallet_vol and df_full are already loaded in the preceding cells
print(f"Trades loaded: {len(df_full):,}")
print(f"Wallets: {df_full['wallet'].nunique():,}")


Trades loaded: 14,250,603


Wallets: 4,082


## Wallet Quality Scoring

In [19]:

# Continuous wallet quality score

def compute_quality_score(wv):
    scores = pd.DataFrame(index=wv.index)
    for col in QUALITY_METRICS:
        scores[col] = wv[col].rank(pct=True)
    for col in QUALITY_PENALTIES:
        scores[f'1_{col}'] = 1.0 - wv[col].rank(pct=True)
    score_cols = list(QUALITY_METRICS) + [f'1_{c}' for c in QUALITY_PENALTIES]
    wv['quality_score'] = scores[score_cols].mean(axis=1).clip(0, 1)
    return wv

wallet_vol = compute_quality_score(wallet_vol)
quality_threshold = wallet_vol['quality_score'].quantile(1.0 - QW_TOP_PCT)

quality_wallets = set(wallet_vol.loc[
    wallet_vol['quality_score'] >= quality_threshold, 'wallet'
])
print(f"Quality wallets (top {QW_TOP_PCT:.0%}): {len(quality_wallets)}")
print(f"  Threshold quality_score: {quality_threshold:.4f}")

# Quick stats
qw_stats = wallet_vol.loc[wallet_vol['wallet'].isin(quality_wallets), 'quality_score']
print(f"  Quality score range: {qw_stats.min():.4f} - {qw_stats.max():.4f}")
print(f"  Median quality score: {qw_stats.median():.4f}")


Quality wallets (top 20%): 717
  Threshold quality_score: 0.6372


  Quality score range: 0.6373 - 0.8392
  Median quality score: 0.6993


## Signal Computation

### Step 1: Market-level Window Aggregations

Compute per-(condition_id, outcome, 15min window) aggregations from ALL trades. These are merged back to each trade using the previous window's values to prevent look-ahead bias.

In [20]:

# Market-level window aggregations (ALL trades, both sides)
# Uses separate groupbys to avoid lambda issues with observed=True.

WINDOW = f'{SIGNAL_WINDOW_MIN}min'

df_full['dt_window'] = df_full['dt'].dt.floor(WINDOW)

# All-trade window metrics
window_all = df_full.groupby(
    ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
).agg(
    mkt_trade_count=('wallet', 'size'),
    mkt_wallet_count=('wallet', 'nunique'),
    mkt_avg_price=('price', 'mean'),
    mkt_price_std=('price', 'std'),
    mkt_price_min=('price', 'min'),
    mkt_price_max=('price', 'max'),
).reset_index()

# Side-specific volume
buy_vol = df_full[df_full['side'] == 'BUY'].groupby(
    ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
)['usdc_amount'].sum().reset_index().rename(columns={'usdc_amount': 'mkt_buy_vol'})

sell_vol = df_full[df_full['side'] == 'SELL'].groupby(
    ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
)['usdc_amount'].sum().reset_index().rename(columns={'usdc_amount': 'mkt_sell_vol'})

# Side-specific wallet counts
buy_wallets = df_full[df_full['side'] == 'BUY'].groupby(
    ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
)['wallet'].nunique().reset_index().rename(columns={'wallet': 'mkt_buy_wallet_count'})

sell_wallets = df_full[df_full['side'] == 'SELL'].groupby(
    ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
)['wallet'].nunique().reset_index().rename(columns={'wallet': 'mkt_sell_wallet_count'})

# Side-specific trade counts
buy_trades = df_full[df_full['side'] == 'BUY'].groupby(
    ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
)['wallet'].size().reset_index().rename(columns={'wallet': 'mkt_buy_trade_count'})

sell_trades = df_full[df_full['side'] == 'SELL'].groupby(
    ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
)['wallet'].size().reset_index().rename(columns={'wallet': 'mkt_sell_trade_count'})

# Merge all
window_market = window_all.merge(buy_vol, on=['condition_id', 'outcome', 'dt_window'], how='left')
window_market = window_market.merge(sell_vol, on=['condition_id', 'outcome', 'dt_window'], how='left')
window_market = window_market.merge(buy_wallets, on=['condition_id', 'outcome', 'dt_window'], how='left')
window_market = window_market.merge(sell_wallets, on=['condition_id', 'outcome', 'dt_window'], how='left')
window_market = window_market.merge(buy_trades, on=['condition_id', 'outcome', 'dt_window'], how='left')
window_market = window_market.merge(sell_trades, on=['condition_id', 'outcome', 'dt_window'], how='left')

# Fill NaN
for c in ['mkt_buy_vol', 'mkt_sell_vol', 'mkt_buy_wallet_count', 'mkt_sell_wallet_count',
          'mkt_buy_trade_count', 'mkt_sell_trade_count']:
    window_market[c] = window_market[c].fillna(0)

# Volume derivatives
window_market['mkt_net_vol'] = window_market['mkt_buy_vol'] - window_market['mkt_sell_vol']
window_market['mkt_bs_ratio'] = window_market['mkt_buy_vol'] / window_market['mkt_sell_vol'].clip(lower=1.0)
window_market['mkt_trade_imbalance'] = (
    (window_market['mkt_buy_vol'] - window_market['mkt_sell_vol'])
    / (window_market['mkt_buy_vol'] + window_market['mkt_sell_vol']).clip(lower=1.0)
)
window_market['mkt_price_range'] = 2.0 * window_market['mkt_price_std'].fillna(0)

# Wallet disagreement signals
tot_w = window_market['mkt_wallet_count'].clip(lower=1)
window_market['mkt_wallet_buy_share'] = window_market['mkt_buy_wallet_count'] / tot_w
window_market['mkt_wallet_sell_share'] = window_market['mkt_sell_wallet_count'] / tot_w

p = window_market['mkt_wallet_buy_share'].clip(0.001, 0.999)
window_market['mkt_wallet_side_entropy'] = -(
    p * np.log2(p) + (1 - p) * np.log2(1 - p)
)

# Trade intensity
window_market['mkt_wallet_trade_intensity'] = (
    window_market['mkt_trade_count'] / window_market['mkt_wallet_count'].clip(lower=1)
)

# Price CV (volatility per unit price)
window_market['mkt_price_cv'] = window_market['mkt_price_std'].fillna(0) / window_market['mkt_avg_price'].clip(lower=1e-9)

# Shift by one window to prevent look-ahead
window_market['dt_window_prev'] = window_market['dt_window'] - pd.Timedelta(minutes=SIGNAL_WINDOW_MIN)

# Pre-compute previous-window values for growth signals
# (before shift, so _prev is the true prior window for each market-outcome)
wm_sorted = window_market.sort_values(['condition_id', 'outcome', 'dt_window'])
prev_cols = ['mkt_wallet_count', 'mkt_price_std', 'mkt_avg_price',
             'mkt_trade_count', 'mkt_buy_vol', 'mkt_sell_vol']
for col in prev_cols:
    window_market[f'{col}_prev'] = wm_sorted.groupby(
        ['condition_id', 'outcome'], sort=False
    )[col].shift(1)

# Growth signals (next_window / current_window - 1)
eps = 1e-12
window_market['mkt_wallet_count_growth'] = (
    window_market['mkt_wallet_count'] / window_market['mkt_wallet_count_prev'].clip(lower=eps) - 1
)
window_market['mkt_price_volatility_ratio'] = (
    window_market['mkt_price_std'] / window_market['mkt_price_std_prev'].clip(lower=eps)
)
window_market['mkt_volume_change'] = (
    (window_market['mkt_buy_vol'] + window_market['mkt_sell_vol'])
    / (window_market['mkt_buy_vol_prev'] + window_market['mkt_sell_vol_prev']).clip(lower=eps) - 1
)
window_market['mkt_price_momentum'] = (
    window_market['mkt_avg_price'] / window_market['mkt_avg_price_prev'].clip(lower=eps) - 1
)
window_market['mkt_trade_count_growth'] = (
    window_market['mkt_trade_count'] / window_market['mkt_trade_count_prev'].clip(lower=eps) - 1
)
window_market['mkt_buy_vol_growth'] = (
    window_market['mkt_buy_vol'] / window_market['mkt_buy_vol_prev'].clip(lower=eps) - 1
)

print(f"Market window stats: {len(window_market):,} windows")
print(f"  Columns: {[c for c in window_market.columns if c.startswith('mkt_')]}")


Market window stats: 3,905,238 windows
  Columns: ['mkt_trade_count', 'mkt_wallet_count', 'mkt_avg_price', 'mkt_price_std', 'mkt_price_min', 'mkt_price_max', 'mkt_buy_vol', 'mkt_sell_vol', 'mkt_buy_wallet_count', 'mkt_sell_wallet_count', 'mkt_buy_trade_count', 'mkt_sell_trade_count', 'mkt_net_vol', 'mkt_bs_ratio', 'mkt_trade_imbalance', 'mkt_price_range', 'mkt_wallet_buy_share', 'mkt_wallet_sell_share', 'mkt_wallet_side_entropy', 'mkt_wallet_trade_intensity', 'mkt_price_cv', 'mkt_wallet_count_prev', 'mkt_price_std_prev', 'mkt_avg_price_prev', 'mkt_trade_count_prev', 'mkt_buy_vol_prev', 'mkt_sell_vol_prev', 'mkt_wallet_count_growth', 'mkt_price_volatility_ratio', 'mkt_volume_change', 'mkt_price_momentum', 'mkt_trade_count_growth', 'mkt_buy_vol_growth']


In [21]:

# Quality-wallet window aggregations

if len(quality_wallets) == 0:
    print("No quality wallets, skipping QW signals")
    qw_window = pd.DataFrame()
else:
    qw_trades = df_full[df_full['wallet'].isin(quality_wallets)].copy()
    qw_trades['quality_score'] = qw_trades['wallet'].map(
        wallet_vol.set_index('wallet')['quality_score'].to_dict()
    )
    qw_trades['qw_weighted_vol'] = qw_trades['quality_score'] * qw_trades['usdc_amount']

    qw_window = qw_trades.groupby(
        ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
    ).agg(
        qw_score_sum=('quality_score', 'sum'),
        qw_score_mean=('quality_score', 'mean'),
        qw_trade_count=('wallet', 'size'),
        qw_wallet_count=('wallet', 'nunique'),
        qw_weighted_vol=('qw_weighted_vol', 'sum'),
        qw_raw_vol=('usdc_amount', 'sum'),
        qw_avg_price=('price', 'mean'),
    ).reset_index()

    # Top-20% quality wallets count
    top_qw_threshold = wallet_vol['quality_score'].quantile(0.80)
    top_qw_wallets = set(wallet_vol.loc[
        wallet_vol['quality_score'] >= top_qw_threshold, 'wallet'
    ])
    qw_trades_top = qw_trades[qw_trades['wallet'].isin(top_qw_wallets)]
    qw_top_count = qw_trades_top.groupby(
        ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
    )['wallet'].nunique().reset_index()
    qw_top_count = qw_top_count.rename(columns={'wallet': 'qw_top_count'})

    qw_window = qw_window.merge(qw_top_count, on=['condition_id', 'outcome', 'dt_window'], how='left')

    # Side-specific quality wallet counts
    qw_buy_wallets = qw_trades[qw_trades['side'] == 'BUY'].groupby(
        ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
    )['wallet'].nunique().reset_index().rename(columns={'wallet': 'qw_buy_wallet_count'})

    qw_sell_wallets = qw_trades[qw_trades['side'] == 'SELL'].groupby(
        ['condition_id', 'outcome', 'dt_window'], sort=False, observed=True
    )['wallet'].nunique().reset_index().rename(columns={'wallet': 'qw_sell_wallet_count'})

    qw_window = qw_window.merge(qw_buy_wallets, on=['condition_id', 'outcome', 'dt_window'], how='left')
    qw_window = qw_window.merge(qw_sell_wallets, on=['condition_id', 'outcome', 'dt_window'], how='left')
    qw_window['qw_buy_wallet_count'] = qw_window['qw_buy_wallet_count'].fillna(0)
    qw_window['qw_sell_wallet_count'] = qw_window['qw_sell_wallet_count'].fillna(0)

    # Quality wallet net direction
    tot_qw_w = qw_window['qw_wallet_count'].clip(lower=1)
    qw_window['qw_buy_share'] = qw_window['qw_buy_wallet_count'] / tot_qw_w

    # Quality wallet net wallet imbalance
    qw_window['qw_net_wallet_imbalance'] = (
        ((qw_window['qw_buy_wallet_count'] - qw_window['qw_sell_wallet_count'])
         / tot_qw_w).fillna(0)
    )

    # Shift
    qw_window['dt_window_prev'] = qw_window['dt_window'] - pd.Timedelta(minutes=SIGNAL_WINDOW_MIN)
    print(f"Quality window stats: {len(qw_window):,} windows")


Quality window stats: 1,641,637 windows


In [22]:

# Quality wallet proximity (merge_asof, nearest-in-past)

if len(quality_wallets) == 0:
    print("No quality wallets, skipping QW proximity")
else:
    if 'qw_wallet_v2' not in df_full.columns:
        qw_buys = df_full[
            df_full['wallet'].isin(quality_wallets) & (df_full['side'] == 'BUY')
        ].copy()
        qw_buys['wallet_qscore'] = qw_buys['wallet'].map(
            wallet_vol.set_index('wallet')['quality_score'].to_dict()
        )
        qw_buys = qw_buys.rename(columns={
            'dt': 'dt_qw', 'wallet': 'qw_wallet2', 'usdc_amount': 'qw_usdc2'
        })[['dt_qw', 'qw_wallet2', 'qw_usdc2', 'wallet_qscore', 'condition_id', 'outcome']].sort_values('dt_qw')

        df_full = pd.merge_asof(
            df_full.sort_values('dt'),
            qw_buys,
            left_on='dt', right_on='dt_qw',
            by=['condition_id', 'outcome'],
            direction='backward',
            tolerance=pd.Timedelta(minutes=SIGNAL_WINDOW_MIN),
            allow_exact_matches=False,
            suffixes=('', '_qw'),
        )
        print("  QW proximity columns added to df_full")
    else:
        print("  QW proximity already computed")


  QW proximity columns added to df_full


### Step 2: VWAP Deviation

In [23]:

# Define active wallet set (used by VWAP + broad sample)
wallet_min_trades = wallet_vol[wallet_vol['trade_count'] >= MIN_WALLET_TRADES]['wallet']
print(f"Wallets with {MIN_WALLET_TRADES}+ trades: {len(wallet_min_trades)}")


Wallets with 100+ trades: 1207


In [24]:

# Wallet subsets for VWAP variants

profitable_buyers = set(wallet_vol[wallet_vol['buy_roi'] > 0.05]['wallet'])
profitable_sellers = set(wallet_vol[wallet_vol['sell_roi'] > 0.05]['wallet'])
print(f"Profitable buyers (buy_roi>0.05): {len(profitable_buyers)}")
print(f"Profitable sellers (sell_roi>0.05): {len(profitable_sellers)}")


Profitable buyers (buy_roi>0.05): 1282
Profitable sellers (sell_roi>0.05): 443


In [25]:

# Multi-source VWAP variants

def compute_bucket_vwap(trades_df, bucket_min=VWAP_BUCKET_MIN):
    df = trades_df.copy()
    df['dt_bucket'] = df['dt'].dt.floor(f'{bucket_min}min')
    df['price_vol'] = df['price'] * df['quantity']
    result = df.groupby(
        ['condition_id', 'outcome', 'dt_bucket'], sort=False, observed=True
    ).agg(
        vwap_price=('price_vol', 'sum'),
        total_qty=('quantity', 'sum'),
        vwap_vol=('usdc_amount', 'sum'),
    ).reset_index()
    result['vwap'] = result['vwap_price'] / result['total_qty'].clip(lower=1e-12)
    result['dt_bucket_prev'] = result['dt_bucket'] - pd.Timedelta(minutes=bucket_min)
    result = result.drop(columns=['vwap_price', 'total_qty'])
    return result

bucket_vwaps = {}

# 1. All-BUY VWAP
all_buys = df_full[df_full['side'] == 'BUY']
bucket_vwaps['allbuy'] = compute_bucket_vwap(all_buys)
print(f"All-buy VWAP: {len(bucket_vwaps['allbuy']):,} buckets")

# 2. All-SELL VWAP
all_sells = df_full[df_full['side'] == 'SELL']
bucket_vwaps['allsell'] = compute_bucket_vwap(all_sells)
print(f"All-sell VWAP: {len(bucket_vwaps['allsell']):,} buckets")

# 3. All-trade VWAP
all_trades = df_full
bucket_vwaps['all'] = compute_bucket_vwap(all_trades)
print(f"All-trade VWAP: {len(bucket_vwaps['all']):,} buckets")

# 4. Profitable-buyer BUY VWAP
pb_buys = df_full[df_full['wallet'].isin(profitable_buyers) & (df_full['side'] == 'BUY')]
bucket_vwaps['pbbuy'] = compute_bucket_vwap(pb_buys)
print(f"Profitable-buyer BUY VWAP: {len(bucket_vwaps['pbbuy']):,} buckets")

# 5. Profitable-seller SELL VWAP
ps_sells = df_full[df_full['wallet'].isin(profitable_sellers) & (df_full['side'] == 'SELL')]
bucket_vwaps['pssell'] = compute_bucket_vwap(ps_sells)
print(f"Profitable-seller SELL VWAP: {len(bucket_vwaps['pssell']):,} buckets")

# 6. Quality-wallet BUY VWAP
qw_buys = df_full[df_full['wallet'].isin(quality_wallets) & (df_full['side'] == 'BUY')]
bucket_vwaps['qwbuy'] = compute_bucket_vwap(qw_buys)
print(f"Quality-wallet BUY VWAP: {len(bucket_vwaps['qwbuy']):,} buckets")

# 7. Copy-wallet BUY VWAP (current approach, wallet_min_trades)
cwl_buys = df_full[df_full['wallet'].isin(wallet_min_trades) & (df_full['side'] == 'BUY')]
bucket_vwaps['copybuy'] = compute_bucket_vwap(cwl_buys)
print(f"Copy-wallet BUY VWAP: {len(bucket_vwaps['copybuy']):,} buckets")

print(f"\nTotal VWAP variants: {len(bucket_vwaps)}")


All-buy VWAP: 4,591,898 buckets


All-sell VWAP: 1,723,432 buckets


All-trade VWAP: 5,310,131 buckets


Profitable-buyer BUY VWAP: 1,677,178 buckets


Profitable-seller SELL VWAP: 178,607 buckets


Quality-wallet BUY VWAP: 1,577,474 buckets


Copy-wallet BUY VWAP: 4,518,146 buckets

Total VWAP variants: 7


### Step 3: Create Broad Sample

Now that df_full is fully annotated, sample 200K BUY trades from active wallets for signal discovery.

In [26]:

# Broad random sample of BUY trades from wallets with 100+ trades
# NOTE: df_full must already be fully annotated (window stats, QW proximity, VWAP)

buy_mask = (
    df_full['wallet'].isin(wallet_min_trades)
    & (df_full['side'] == 'BUY')
)
all_buys = df_full[buy_mask].copy()
print(f"Total BUY trades by these wallets: {len(all_buys):,}")

rng = np.random.RandomState(42)
if len(all_buys) > BROAD_SAMPLE_SIZE:
    sample_idx = rng.choice(len(all_buys), BROAD_SAMPLE_SIZE, replace=False)
    broad_sample = all_buys.iloc[sample_idx].copy()
else:
    broad_sample = all_buys.copy()
print(f"Broad sample: {len(broad_sample):,} trades")

# Split into train/val/test by time
dates = sorted(broad_sample['dt'].unique())
n = len(dates)
train_cutoff = dates[int(n * 0.4)]
val_cutoff = dates[int(n * 0.7)]
print(f"Train cutoff: {train_cutoff}, Val cutoff: {val_cutoff}")

s_train = broad_sample[broad_sample['dt'] < train_cutoff].copy()
s_val = broad_sample[
    (broad_sample['dt'] >= train_cutoff) & (broad_sample['dt'] < val_cutoff)
].copy()
s_test = broad_sample[broad_sample['dt'] >= val_cutoff].copy()
print(f"  Train: {len(s_train):,}  Val: {len(s_val):,}  Test: {len(s_test):,}")


Total BUY trades by these wallets: 10,645,892


Broad sample: 200,000 trades


Train cutoff: 2026-05-30 04:04:45+00:00, Val cutoff: 2026-06-26 06:35:43+00:00
  Train: 82,505  Val: 58,977  Test: 58,518


### Step 4: Merge Signals into Sample

In [27]:

# Merge all window signals into sample splits

def apply_vwap_signal(df_c, vwap_df, name):
    """Merge VWAP variant and compute deviation + signed signal."""
    vwap_col = f'vwap_{name}'
    right = vwap_df[['condition_id', 'outcome', 'dt_bucket_prev', 'vwap']].rename(
        columns={'vwap': vwap_col, 'dt_bucket_prev': 'dt_bucket'}
    )
    df_c = df_c.merge(
        right,
        on=['condition_id', 'outcome', 'dt_bucket'],
        how='left',
    )
    dev_col = f'sig_vwap_{name}_dev'
    sig_col = f'sig_vwap_{name}_signed'
    df_c[dev_col] = np.where(
        df_c[vwap_col].notna() & (df_c[vwap_col] > 0),
        (df_c['price'] / df_c[vwap_col]) - 1.0, np.nan,
    )
    df_c[sig_col] = np.where(
        df_c[dev_col].notna(), -df_c[dev_col], np.nan,
    )
    return df_c.drop(columns=[vwap_col])

def merge_window_signals(df_c, window_market, qw_window, bucket_vwaps):
    df_c = df_c.copy()
    df_c['dt_window'] = df_c['dt'].dt.floor(f'{SIGNAL_WINDOW_MIN}min')
    df_c['dt_window_prev'] = df_c['dt_window'] - pd.Timedelta(minutes=SIGNAL_WINDOW_MIN)
    df_c['dt_bucket'] = df_c['dt'].dt.floor(f'{VWAP_BUCKET_MIN}min')

    # Market window signals (previous window)
    win_cols = ['condition_id', 'outcome', 'dt_window_prev']
    mkt_cols = [
        'mkt_trade_count', 'mkt_wallet_count', 'mkt_buy_vol', 'mkt_sell_vol',
        'mkt_net_vol', 'mkt_bs_ratio', 'mkt_trade_imbalance',
        'mkt_avg_price', 'mkt_price_std', 'mkt_price_range',
        'mkt_buy_wallet_count', 'mkt_sell_wallet_count',
        'mkt_buy_trade_count', 'mkt_sell_trade_count',
        'mkt_wallet_buy_share', 'mkt_wallet_sell_share',
        'mkt_wallet_side_entropy', 'mkt_wallet_trade_intensity',
        'mkt_price_cv',
        'mkt_wallet_count_growth', 'mkt_price_volatility_ratio',
        'mkt_volume_change', 'mkt_price_momentum',
        'mkt_trade_count_growth', 'mkt_buy_vol_growth',
    ]
    df_c = df_c.merge(
        window_market[win_cols + mkt_cols],
        left_on=['condition_id', 'outcome', 'dt_window'],
        right_on=win_cols,
        how='left', suffixes=('', '_mkt')
    )

    # Quality window signals
    if len(qw_window) > 0:
        qw_cols = [
            'qw_score_sum', 'qw_score_mean', 'qw_trade_count',
            'qw_wallet_count', 'qw_weighted_vol', 'qw_raw_vol', 'qw_top_count',
            'qw_buy_wallet_count', 'qw_sell_wallet_count',
            'qw_buy_share', 'qw_net_wallet_imbalance', 'qw_avg_price',
        ]
        df_c = df_c.merge(
            qw_window[win_cols + qw_cols],
            left_on=['condition_id', 'outcome', 'dt_window'],
            right_on=win_cols,
            how='left', suffixes=('', '_qw')
        )

    # QW price premium: how quality wallet avg price differs from market avg
    if len(qw_window) > 0 and 'qw_avg_price' in df_c.columns:
        df_c['qw_price_premium'] = np.where(
            df_c['qw_avg_price'].notna() & (df_c['mkt_avg_price'] > 0),
            (df_c['qw_avg_price'] / df_c['mkt_avg_price']) - 1.0, np.nan,
        )

    # QW proximity
    df_c['sig_qw_proximity'] = df_c['qw_wallet2'].notna().astype(float)
    df_c['sig_qw_prox_volume'] = df_c['qw_usdc2'].fillna(0.0)
    df_c['sig_qw_prox_qscore'] = df_c['wallet_qscore'].fillna(0.0)

    # VWAP variants
    for name, vwap_df in bucket_vwaps.items():
        df_c = apply_vwap_signal(df_c, vwap_df, name)

    # CS-rank on copybuy VWAP signal (our primary variant)
    df_c['sig_vwap_csrank'] = cs_rank(df_c['sig_vwap_copybuy_signed'].fillna(0.0), df_c['dt'].dt.date)

    return df_c

for name, df_c in [('Train', s_train), ('Val', s_val), ('Test', s_test)]:
    updated = merge_window_signals(df_c, window_market, qw_window, bucket_vwaps)
    if name == 'Train': s_train = updated
    elif name == 'Val': s_val = updated
    elif name == 'Test': s_test = updated
    print(f"{name}: {len(df_c)} -> {len(updated)} rows")

# Define signal columns
raw_signal_cols = [
    'sig_qw_proximity', 'sig_qw_prox_volume', 'sig_qw_prox_qscore',
    'sig_vwap_csrank',
]
vwap_variant_names = ['allbuy', 'allsell', 'all', 'pbbuy', 'pssell', 'qwbuy', 'copybuy']
vwap_variant_cols = []
for vn in vwap_variant_names:
    vwap_variant_cols += [f'sig_vwap_{vn}_dev', f'sig_vwap_{vn}_signed']

market_signal_cols = [
    'mkt_trade_count', 'mkt_wallet_count', 'mkt_net_vol', 'mkt_bs_ratio',
    'mkt_trade_imbalance', 'mkt_avg_price', 'mkt_price_range',
    # New market signals
    'mkt_wallet_buy_share', 'mkt_wallet_side_entropy',
    'mkt_wallet_trade_intensity', 'mkt_price_cv',
]

qw_window_cols = [
    'qw_score_sum', 'qw_score_mean', 'qw_trade_count', 'qw_wallet_count',
    'qw_weighted_vol', 'qw_raw_vol', 'qw_top_count',
    # New QW signals
    'qw_buy_share', 'qw_net_wallet_imbalance', 'qw_avg_price', 'qw_price_premium',
]

growth_signal_cols = [
    'mkt_wallet_count_growth', 'mkt_price_volatility_ratio',
    'mkt_volume_change', 'mkt_price_momentum',
    'mkt_trade_count_growth', 'mkt_buy_vol_growth',
]

signal_cols = raw_signal_cols + vwap_variant_cols + market_signal_cols + qw_window_cols + growth_signal_cols
signal_cols = [c for c in signal_cols if c in s_train.columns]

# Cross-sectional rank each market + qw + growth signal per day for comparability
for col in market_signal_cols + qw_window_cols + growth_signal_cols:
    if col in s_train.columns:
        s_train[f'cs_{col}'] = cs_rank(s_train[col].fillna(0.0), s_train['dt'].dt.date)
        s_val[f'cs_{col}'] = cs_rank(s_val[col].fillna(0.0), s_val['dt'].dt.date)
        s_test[f'cs_{col}'] = cs_rank(s_test[col].fillna(0.0), s_test['dt'].dt.date)

# Also CS-rank each VWAP variant
for vn in vwap_variant_names:
    col = f'sig_vwap_{vn}_signed'
    if col in s_train.columns:
        cs_col = f'cs_vwap_{vn}'
        s_train[cs_col] = cs_rank(s_train[col].fillna(0.0), s_train['dt'].dt.date)
        s_val[cs_col] = cs_rank(s_val[col].fillna(0.0), s_val['dt'].dt.date)
        s_test[cs_col] = cs_rank(s_test[col].fillna(0.0), s_test['dt'].dt.date)

cs_signal_cols = (
    [f'cs_{c}' for c in market_signal_cols + qw_window_cols + growth_signal_cols if f'cs_{c}' in s_train.columns]
    + [f'cs_vwap_{vn}' for vn in vwap_variant_names if f'cs_vwap_{vn}' in s_train.columns]
)
all_signal_cols = signal_cols + cs_signal_cols
all_signal_cols = [c for c in all_signal_cols if c in s_train.columns]

print(f"\nTotal signals: {len(all_signal_cols)}")
for c in all_signal_cols:
    nnan = s_val[c].notna().sum()
    print(f"  {c:35s}: non-null={nnan:>8,}  mean={s_val[c].mean():.4f}")


Train: 82505 -> 82505 rows


Val: 58977 -> 58977 rows


Test: 58518 -> 58518 rows



Total signals: 81
  sig_qw_proximity                   : non-null=  58,977  mean=0.3869
  sig_qw_prox_volume                 : non-null=  58,977  mean=6.4644
  sig_qw_prox_qscore                 : non-null=  58,977  mean=0.2634
  sig_vwap_csrank                    : non-null=  58,977  mean=-0.0000
  sig_vwap_allbuy_dev                : non-null=  25,070  mean=2.0345
  sig_vwap_allbuy_signed             : non-null=  25,070  mean=-2.0345
  sig_vwap_allsell_dev               : non-null=  14,058  mean=2.9270
  sig_vwap_allsell_signed            : non-null=  14,058  mean=-2.9270
  sig_vwap_all_dev                   : non-null=  27,629  mean=1.7339
  sig_vwap_all_signed                : non-null=  27,629  mean=-1.7339
  sig_vwap_pbbuy_dev                 : non-null=  10,867  mean=18.4522
  sig_vwap_pbbuy_signed              : non-null=  10,867  mean=-18.4522
  sig_vwap_pssell_dev                : non-null=   2,883  mean=5.9295
  sig_vwap_pssell_signed             : non-null=   2,883  mean=-

## Signal Evaluation

In [28]:

# IC evaluation for each signal with bootstrapped CIs

print("=" * 90)
print(f"{'Signal':35s} {'Val_IC':>9s} {'Val_pIR':>9s} {'Val_hit':>9s} {'Train_IC':>9s} {'Test_IC':>9s}")
print("=" * 90)

signal_results = {}
for sig in all_signal_cols:
    n_train = s_train[sig].notna().sum()
    n_val = s_val[sig].notna().sum()
    n_test = s_test[sig].notna().sum()
    if min(n_train, n_val, n_test) < 100:
        continue
    ic_val = compute_event_ic(s_val[sig], s_val['copyable_roi']) or 0.0
    ic_train = compute_event_ic(s_train[sig], s_train['copyable_roi']) or 0.0
    ic_test = compute_event_ic(s_test[sig], s_test['copyable_roi']) or 0.0
    hit_val = (s_val[sig].notna() & (s_val['copyable_roi'] > 0)).mean() if s_val[sig].notna().any() else 0.0

    signal_results[sig] = {
        'val_ic': ic_val, 'train_ic': ic_train, 'test_ic': ic_test,
        'val_n': n_val, 'hit_val': hit_val,
    }
    print(f"{sig:35s} {ic_val:9.4f} {ic_val * np.sqrt(n_val):9.4f} {hit_val:9.4f} {ic_train:9.4f} {ic_test:9.4f}")

# Sort by |val_ic|
ranked = sorted(signal_results.items(), key=lambda x: abs(x[1]['val_ic']), reverse=True)
print(f"\n--- Top signals by |Val IC| ---")
for sig, r in ranked[:10]:
    print(f"  {sig:35s}  Val IC={r['val_ic']:.4f}  Train IC={r['train_ic']:.4f}  Test IC={r['test_ic']:.4f}  (n={r['val_n']:,})")

# Signals passing threshold
PASS_IC = 0.010
passing = [sig for sig, r in ranked if abs(r['val_ic']) >= PASS_IC and r['train_ic'] * r['val_ic'] > 0]
print(f"\nSignals with |IC| >= {PASS_IC} AND consistent sign: {len(passing)}")
for sig in passing:
    r = signal_results[sig]
    print(f"  {sig:35s}  Val IC={r['val_ic']:.4f}  Train IC={r['train_ic']:.4f}  Test IC={r['test_ic']:.4f}")


Signal                                 Val_IC   Val_pIR   Val_hit  Train_IC   Test_IC
sig_qw_proximity                       0.0068    1.6440    0.2824    0.0025    0.0057


sig_qw_prox_volume                     0.0073    1.7623    0.2824    0.0009    0.0052


sig_qw_prox_qscore                     0.0062    1.4940    0.2824   -0.0002    0.0053
sig_vwap_csrank                        0.0035    0.8561    0.2824    0.0001    0.0073


sig_vwap_allbuy_dev                    0.0081    1.2749    0.1379    0.0044    0.0009


sig_vwap_allbuy_signed                 0.0011    0.1801    0.1379    0.0113    0.0054
sig_vwap_allsell_dev                   0.0079    0.9328    0.0828   -0.0016    0.0299
sig_vwap_allsell_signed               -0.0108   -1.2770    0.0828    0.0060    0.0006
sig_vwap_all_dev                      -0.0014   -0.2326    0.1525   -0.0016   -0.0057


...

cs_mkt_volume_change                  -0.0023   -0.5696    0.2824    0.0008    0.0073
cs_mkt_price_momentum                 -0.0128   -3.1092    0.2824   -0.0035    0.0010


cs_mkt_trade_count_growth             -0.0009   -0.2267    0.2824   -0.0035   -0.0066
cs_mkt_buy_vol_growth                  0.0068    1.6574    0.2824    0.0049   -0.0095


cs_vwap_allbuy                         0.0012    0.2940    0.2824    0.0037   -0.0111
cs_vwap_allsell                        0.0050    1.2254    0.2824   -0.0029    0.0022
cs_vwap_all                            0.0040    0.9677    0.2824    0.0022    0.0007


cs_vwap_pbbuy                         -0.0022   -0.5267    0.2824    0.0026    0.0030
cs_vwap_pssell                         0.0039    0.9369    0.2824    0.0042    0.0051
cs_vwap_qwbuy                         -0.0065   -1.5879    0.2824   -0.0046    0.0006


cs_vwap_copybuy                        0.0035    0.8561    0.2824    0.0001    0.0073

--- Top signals by |Val IC| ---
  qw_net_wallet_imbalance              Val IC=-0.0157  Train IC=-0.0119  Test IC=-0.0030  (n=22,047)
  sig_vwap_pssell_dev                  Val IC=-0.0144  Train IC=0.0079  Test IC=0.0070  (n=2,883)
  qw_wallet_count                      Val IC=-0.0130  Train IC=0.0067  Test IC=0.0028  (n=22,047)
  qw_top_count                         Val IC=-0.0130  Train IC=0.0067  Test IC=0.0028  (n=22,047)
  cs_mkt_price_momentum                Val IC=-0.0128  Train IC=-0.0035  Test IC=0.0010  (n=58,977)
  sig_vwap_allsell_signed              Val IC=-0.0108  Train IC=0.0060  Test IC=0.0006  (n=14,058)
  cs_mkt_wallet_side_entropy           Val IC=0.0103  Train IC=-0.0035  Test IC=-0.0079  (n=58,977)
  sig_vwap_copybuy_signed              Val IC=0.0103  Train IC=0.0058  Test IC=0.0095  (n=24,923)
  cs_qw_trade_count                    Val IC=0.0101  Train IC=0.0027  Test IC=-0.0086 

In [29]:

# Forward selection of signals

def conditional_ic(target_col, candidate, baseline_cols, df):
    """IC of candidate after orthogonalizing wrt baseline_cols."""
    if not baseline_cols:
        return compute_event_ic(df[candidate], df[target_col]) or 0.0
    # Orthogonalize candidate against baseline using cross-sectional ranks
    # Simple approach: residual of linear regression
    X = df[baseline_cols].fillna(0).values
    y = df[candidate].fillna(0).values
    try:
        beta = np.linalg.lstsq(X, y, rcond=None)[0]
        residual = y - X @ beta
    except np.linalg.LinAlgError:
        return 0.0
    return compute_event_ic(pd.Series(residual, index=df.index), df[target_col]) or 0.0

active_signals = list(passing)  # signals that passed initial screen
if len(active_signals) < 2:
    print(f"Too few signals ({len(active_signals)}) for forward selection")
    selected = active_signals
else:
    # Sort by |val_ic|, pick best first
    selected = [active_signals[0]]
    remaining = list(active_signals[1:])

    print(f"Forward selection starting with: {selected[0]}")
    print(f"{'Step':>5s} {'Selected':30s} {'Cond_IC':>9s} {'Running Count':>13s}")
    print("-" * 60)

    for step in range(min(5, len(remaining))):
        best_cond = -float('inf')
        best_sig = None
        for sig in remaining:
            cond = conditional_ic('copyable_roi', sig, selected, s_val)
            if cond > best_cond:
                best_cond = cond
                best_sig = sig
        if best_sig is None or best_cond <= 0.001:
            break
        selected.append(best_sig)
        remaining.remove(best_sig)
        print(f"{step + 1:5d} {best_sig:30s} {best_cond:9.4f} ({len(selected):3d} total)")

    print(f"\nSelected {len(selected)} signals: {selected}")

# If no signals pass, use top 3 by val_ic (regardless of sign consistency)
if not selected and ranked:
    selected = [sig for sig, r in ranked[:3]]
    print(f"No signals pass threshold, using top 3: {selected}")


Forward selection starting with: qw_net_wallet_imbalance
 Step Selected                         Cond_IC Running Count
------------------------------------------------------------
    1 sig_vwap_pssell_signed            0.0037 (  2 total)



Selected 2 signals: ['qw_net_wallet_imbalance', 'sig_vwap_pssell_signed']


## Parameter Tuning

In [30]:

# === Tune VWAP bucket size ===

def tune_vwap_bucket(bucket_min, df_sample, target_col='copyable_roi'):
    """Compute VWAP signal at given bucket size and return IC."""
    copy_buys = df_full[
        df_full['wallet'].isin(wallet_min_trades) & (df_full['side'] == 'BUY')
    ].copy()
    copy_buys['price_vol'] = copy_buys['price'] * copy_buys['quantity']
    copy_buys['dt_bucket'] = copy_buys['dt'].dt.floor(f'{bucket_min}min')
    
    bv = copy_buys.groupby(
        ['condition_id', 'outcome', 'dt_bucket'], sort=False, observed=True
    ).agg(
        vwap_price=('price_vol', 'sum'),
        total_qty=('quantity', 'sum'),
    ).reset_index()
    bv['vwap'] = bv['vwap_price'] / bv['total_qty'].clip(lower=1e-12)
    bv['dt_bucket_prev'] = bv['dt_bucket'] - pd.Timedelta(minutes=bucket_min)
    
    result = df_sample.copy()
    result['dt_bucket'] = result['dt'].dt.floor(f'{bucket_min}min')
    result = result.merge(
        bv[['condition_id', 'outcome', 'dt_bucket_prev', 'vwap']].rename(
            columns={'dt_bucket_prev': 'dt_bucket'}),
        on=['condition_id', 'outcome', 'dt_bucket'],
        how='left',
    )
    dev = np.where(result['vwap'].notna() & (result['vwap'] > 0),
                   (result['price'] / result['vwap']) - 1.0, np.nan)
    sig = pd.Series(np.where(np.isfinite(dev), -dev, np.nan), index=result.index)
    return compute_event_ic(sig, result[target_col]) or 0.0

bucket_sizes = [1, 2, 3, 5, 7, 10, 15, 20, 30]
print(f"\nTuning VWAP bucket size (copybuy VWAP):")
print(f"{'bucket_min':>12s} {'Val_IC':>9s} {'Train_IC':>9s} {'Test_IC':>9s} {'n_val':>8s}")
print("-" * 50)

vwap_tune_results = []
for bm in bucket_sizes:
    ic_val = tune_vwap_bucket(bm, s_val)
    ic_train = tune_vwap_bucket(bm, s_train)
    ic_test = tune_vwap_bucket(bm, s_test)
    n_val = int(s_val['dt'].dt.floor(f'{bm}min').notna().sum())
    vwap_tune_results.append({'bucket_min': bm, 'val_ic': ic_val,
                              'train_ic': ic_train, 'test_ic': ic_test, 'n_val': n_val})
    print(f"{bm:12d} {ic_val:9.4f} {ic_train:9.4f} {ic_test:9.4f} {n_val:8d}")

best_vwap = max(vwap_tune_results, key=lambda r: abs(r['val_ic']))
print(f"\nBest VWAP bucket: {best_vwap['bucket_min']}min (Val IC={best_vwap['val_ic']:.4f})")
# Same for train consistency
train_consistent = [r for r in vwap_tune_results if r['train_ic'] * r['val_ic'] > 0]
if train_consistent:
    best_consistent = max(train_consistent, key=lambda r: abs(r['val_ic']))
    print(f"Best (consistent sign): {best_consistent['bucket_min']}min (Val IC={best_consistent['val_ic']:.4f})")



Tuning VWAP bucket size (copybuy VWAP):
  bucket_min    Val_IC  Train_IC   Test_IC    n_val
--------------------------------------------------


           1    0.0083    0.0016    0.0020    58977


           2    0.0148   -0.0014   -0.0091    58977


           3    0.0017    0.0000   -0.0034    58977


           5    0.0103    0.0058    0.0095    58977


           7    0.0051    0.0019   -0.0068    58977


          10    0.0077   -0.0049   -0.0006    58977


          15    0.0118   -0.0046    0.0081    58977


          20   -0.0007   -0.0079    0.0046    58977


          30   -0.0015   -0.0006   -0.0027    58977

Best VWAP bucket: 2min (Val IC=0.0148)
Best (consistent sign): 5min (Val IC=0.0103)


## Apply to Copy Trades

In [31]:

# Apply winning signals to copy-universe trades

# Define copy universe (less restrictive than before)
copy_mask = (
    (wallet_vol['copyable_pnl'] > 0)
    & (wallet_vol['copyable_roi'].fillna(0) >= 0.02)
    & (wallet_vol['num_buckets'] >= 20)
    & (wallet_vol['trade_count'] >= MIN_WALLET_TRADES)
    & (wallet_vol['max_drawdown_to_pnl'].fillna(1.0) <= 0.5)
)
copy_wallets = set(wallet_vol.loc[copy_mask, 'wallet'])
print(f"Copy wallets: {len(copy_wallets)}")

candidate_mask = df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
candidate_trades = df_full[candidate_mask].copy()
print(f"Candidate BUY trades: {len(candidate_trades):,}")

# Split candidate trades into train/val/test
c_dates = sorted(candidate_trades['dt'].unique())
n = len(c_dates)
cc_train = c_dates[int(n * 0.4)]
cc_val = c_dates[int(n * 0.7)]
ct_train = candidate_trades[candidate_trades['dt'] < cc_train].copy()
ct_val = candidate_trades[
    (candidate_trades['dt'] >= cc_train) & (candidate_trades['dt'] < cc_val)
].copy()
ct_test = candidate_trades[candidate_trades['dt'] >= cc_val].copy()
print(f"  C_Train: {len(ct_train):,}  C_Val: {len(ct_val):,}  C_Test: {len(ct_test):,}")

# Merge signals into copy trades using the same function
for name, df_c in [('C_Train', ct_train), ('C_Val', ct_val), ('C_Test', ct_test)]:
    df_c = merge_window_signals(df_c, window_market, qw_window, bucket_vwaps)
    for col in market_signal_cols + qw_window_cols + growth_signal_cols:
        cs_col = f'cs_{col}'
        if col in df_c.columns:
            df_c[cs_col] = cs_rank(df_c[col].fillna(0.0), df_c['dt'].dt.date)
    if name == 'C_Train': ct_train = df_c
    elif name == 'C_Val': ct_val = df_c
    elif name == 'C_Test': ct_test = df_c

# Signal quality report on copy-val
copy_sig_cols = [c for c in selected if c in ct_val.columns]
if len(copy_sig_cols) >= 1:
    print("\nSignal IC on copy-universe trades:")
    for sig in copy_sig_cols:
        ic_val = compute_event_ic(ct_val[sig], ct_val['copyable_roi']) or 0.0
        ic_test = compute_event_ic(ct_test[sig], ct_test['copyable_roi']) or 0.0
        print(f"  {sig:35s}  Val IC={ic_val:.4f}  Test IC={ic_test:.4f}")


Copy wallets: 154


Candidate BUY trades: 481,956


  C_Train: 194,530  C_Val: 146,758  C_Test: 140,668



Signal IC on copy-universe trades:
  qw_net_wallet_imbalance              Val IC=0.0071  Test IC=-0.0017
  sig_vwap_pssell_signed               Val IC=0.0040  Test IC=-0.0373


In [32]:

# Strategy evaluation with composite score on copy trades

copy_sig_cols = [c for c in selected if c in ct_val.columns]
if len(copy_sig_cols) < 1:
    print("No signals available for strategy")
else:
    print(f"Using {len(copy_sig_cols)} signals: {copy_sig_cols}")

    # Composite: equal weight
    for df_c in [ct_train, ct_val, ct_test]:
        df_c['composite'] = df_c[copy_sig_cols].fillna(0.0).sum(axis=1) / len(copy_sig_cols)

    # Baseline: all trades
    all_roi = ct_test['copyable_roi'].mean()
    all_cpnl = ct_test['copyable_pnl'].sum()
    print(f"\nBaseline (all copy trades): Test ROI={all_roi:.4f}  Test CPnL={all_cpnl:.2f}")

    # Strategy at various thresholds
    print(f"\n{'threshold':>10s} {'trades':>8s} {'roi':>10s} {'cpnl':>10s} {'cpnl_diff':>10s}")
    print("-" * 50)
    best_spnl = -float('inf')
    best_thresh = 0.0
    for thresh in np.arange(-0.5, 0.51, 0.1):
        mask = ct_test['composite'].fillna(thresh - 1) >= thresh
        if mask.sum() < 5:
            continue
        roi = ct_test.loc[mask, 'copyable_roi'].mean()
        cpnl = ct_test.loc[mask, 'copyable_pnl'].sum()
        diff = cpnl - all_cpnl * (mask.sum() / len(ct_test))
        print(f"{thresh:10.1f} {mask.sum():8d} {roi:10.4f} {cpnl:10.2f} {diff:10.2f}")
        if cpnl > best_spnl:
            best_spnl = cpnl
            best_thresh = thresh

    print(f"\nBest threshold: {best_thresh:.2f} (CPnL={best_spnl:.2f})")

    # Evaluate best on all splits
    print(f"\n{'Split':>10s} {'threshold':>10s} {'trades':>8s} {'roi':>10s} {'cpnl':>10s} {'all_roi':>10s} {'all_cpnl':>10s}")
    print("-" * 70)
    for label, df_c in [('Train', ct_train), ('Val', ct_val), ('Test', ct_test)]:
        mask = df_c['composite'].fillna(best_thresh - 1) >= best_thresh
        all_roi_split = df_c['copyable_roi'].mean()
        all_cpnl_split = df_c['copyable_pnl'].sum()
        if mask.sum() < 5:
            sig_roi, sig_cpnl = 0.0, 0.0
        else:
            sig_roi = df_c.loc[mask, 'copyable_roi'].mean()
            sig_cpnl = df_c.loc[mask, 'copyable_pnl'].sum()
        print(f"{label:>10s} {best_thresh:10.2f} {mask.sum():8d} {sig_roi:10.4f} {sig_cpnl:10.2f} {all_roi_split:10.4f} {all_cpnl_split:10.2f}")


Using 2 signals: ['qw_net_wallet_imbalance', 'sig_vwap_pssell_signed']

Baseline (all copy trades): Test ROI=0.0188  Test CPnL=16965.65

 threshold   trades        roi       cpnl  cpnl_diff
--------------------------------------------------
      -0.5   140379     0.0207   17693.82     763.02
      -0.4   133737     0.0186   15959.76    -169.96
      -0.3   133254     0.0148   15836.05    -235.42
      -0.2   131411    -0.0209   15003.07    -846.12
      -0.1   129666    -0.0319   11496.32   -4142.41
      -0.0   129119    -0.0356   10415.55   -5157.21
       0.1    34526     0.0185    8466.13    4302.03
       0.2    31338    -0.0029    5468.89    1689.28
       0.3    27679     0.0127    3835.41     497.10
       0.4    25915     0.0090    1933.02   -1192.53
       0.5    25657    -0.0050    1239.90   -1854.54

Best threshold: -0.50 (CPnL=17693.82)

     Split  threshold   trades        roi       cpnl    all_roi   all_cpnl
-------------------------------------------------------------

## Summary


**Results Summary**

We tested signals on a broad random sample of 200K BUY trades from wallets with 100+ trades.
Signals that passed initial screening were applied to copy-universe trades.

Key metrics:
- Number of signals tested
- Number passing IC threshold
- Composite performance on copy trades vs baseline
